# Building a Multi-Agent E-Commerce Support System with AutoGen and Moss

This cookbook demonstrates how to build a collaborative, multi-agent e-commerce support system using **AutoGen** and **Moss**. 

When AutoGen agents need to search for external knowledge, network retrieval speeds can become a massive bottleneck. By coupling AutoGen with Moss, we can load our vector indices straight into local memory, resulting in blazing-fast, sub-10ms context retrieval that keeps multi-agent loops running instantly.

**System Highlights:**
- **Domain Isolation:** Independent vector knowledge bases deployed via Moss, preventing agents from hallucinating cross-domain context.
- **Zero-Latency Tools:** Real-time metrics proving how in-memory semantic search eliminates the compounding latency bottlenecks typical of AutoGen workflows.

**Prerequisites:**
- An OpenAI API Key
- A Moss Project ID and Key (Get yours at [moss.dev](https://www.moss.dev/))

In [12]:
!uv pip install -qU "autogen-agentchat" "autogen-ext[openai]" "moss" "python-dotenv"

## Setup & Core Clients

We initialize our LLM to power the AutoGen agents, and `moss_client` for our semantic engine.


In [13]:
import os
import json
from dotenv import load_dotenv
from moss import MossClient, DocumentInfo, QueryOptions, SearchResult
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.model_context import BufferedChatCompletionContext
load_dotenv()

moss_client = MossClient(
    project_id=os.getenv("MOSS_PROJECT_ID"),
    project_key=os.getenv("MOSS_PROJECT_KEY")
)

model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

## Creating Isolated Knowledge Domains

When building multi-agent systems, it's crucial to prevent context cross-contamination. If a shipping agent is asked about timelines, it shouldn't be searching through product specifications. 

We resolve this by creating three distinct vector indices: `product_catalog_index`, `shipping_policies_index`, and `returns_index`.

Run this Cell once as this will create and store indices in Moss Cloud 


In [14]:
async def setup_indices():
    all_indexes = await moss_client.list_indexes()
    existing_names = {idx.name for idx in all_indexes}
    async def create_index_from_file(filename, index_name):
        if index_name in existing_names:
            print(f"Index: '{index_name}' already exists on Moss Cloud, skipping creation.")
            return
        with open(f"data/{filename}", "r") as f:
            data = json.load(f)
        docs = []
        for i, item in enumerate(data):
            text_repr = ", ".join([f"{k}: {v}" for k, v in item.items()])
            doc_id = item.get("id", f"doc_{i}")
            string_metadata = {k: str(v) for k, v in item.items()}
            docs.append(DocumentInfo(id=doc_id, text=text_repr, metadata=string_metadata))
        await moss_client.create_index(index_name, docs)
        print(f"Indexed {len(docs)} documents into '{index_name}'.")

    await create_index_from_file("product_catalog.json", "product_catalog_index")
    await create_index_from_file("shipping_policies.json", "shipping_policies_index")
    await create_index_from_file("return_policies.json", "returns_index")

# Run once
await setup_indices()

Index: 'product_catalog_index' already exists on Moss Cloud, skipping creation.
Index: 'shipping_policies_index' already exists on Moss Cloud, skipping creation.
Index: 'returns_index' already exists on Moss Cloud, skipping creation.


The `moss_client.load_index()` command is a core feature of the Moss runtime, enabling real-time semantic search directly where your agent lives. By loading the index into your machine's local RAM, it delivers sub-10 ms lookups and eliminates the sequential network overhead typical of cloud-based retrieval. 

In [15]:
async def load_indexes():

    await moss_client.load_index("product_catalog_index")
    await moss_client.load_index("shipping_policies_index")
    await moss_client.load_index("returns_index")

await load_indexes()

## Defining the Agent Tools

We wrap our local Moss queries into explicit async Python functions so they can be registered as tools array parameters on our agents. 

Notice how we extract the `SearchResult.time_taken_ms` property. This lets us benchmark backend performance on the fly. Since we cached the indices locally in the previous step.

In [16]:
# To store moss query latencies across agent execution
latencies = []

async def product_search(search_query: str) -> str:
    res = await moss_client.query("product_catalog_index", search_query, QueryOptions(top_k=6))
    latencies.append(res.time_taken_ms)
    time_display = "N/A" if res.time_taken_ms is None else ("< 1" if res.time_taken_ms == 0 else res.time_taken_ms)
    print(f"\n[Moss Retrieval Time] product_search context retrieved in {time_display} ms")
    if not res.docs:
        return "No relevant information found."
    return "\n\n".join([doc.text for doc in res.docs])

async def shipping_search(search_query: str) -> str:
    res = await moss_client.query("shipping_policies_index", search_query, QueryOptions(top_k=6))
    latencies.append(res.time_taken_ms)
    time_display = "N/A" if res.time_taken_ms is None else ("< 1" if res.time_taken_ms == 0 else res.time_taken_ms)
    print(f"\n[Moss Retrieval Time] shipping_search context retrieved  in {time_display} ms")
    if not res.docs:
        return "No relevant information found."
    return "\n\n".join([doc.text for doc in res.docs])

async def returns_search(search_query: str) -> str:
    res = await moss_client.query("returns_index", search_query, QueryOptions(top_k=6))
    latencies.append(res.time_taken_ms)
    time_display = "N/A" if res.time_taken_ms is None else ("< 1" if res.time_taken_ms == 0 else res.time_taken_ms)
    print(f"\n[Moss Retrieval Time] returns_search context retrieved in {time_display} ms")
    if not res.docs:
        return "No relevant information found."
    return "\n\n".join([doc.text for doc in res.docs])

## Orchestrating the AutoGen Team

Instead of a rigid state machine, we use an intelligent routing pattern. By assigning our `model_client` to the `SelectorGroupChat`, the LLM dynamically reads the prompt and each agent's `system_message` to autonomously determine who should speak next.

In [17]:
product_agent = AssistantAgent(
    "product_agent",
    model_client=model_client,
    tools=[product_search],
    system_message="You are a product specialist. Use the 'product_search' tool to look up catalog information before answering. If an item is out of stock, say so clearly.",
    reflect_on_tool_use=True
)

shipping_agent = AssistantAgent(
    "shipping_agent",
    model_client=model_client,
    tools=[shipping_search],
    system_message="You are a shipping specialist. Use the 'shipping_search' tool to look up shipping policies. For damaged item replacements, confirm eligibility with returns_agent first before quoting delivery times.",
    reflect_on_tool_use=True
)

returns_agent = AssistantAgent(
    "returns_agent",
    model_client=model_client,
    tools=[returns_search],
    system_message="You are a returns specialist. Use the 'returns_search' tool to look up return policies. For damaged items requiring replacement shipping, state clearly that shipping_agent will handle the delivery details.",
    reflect_on_tool_use=True
)

summary_agent = AssistantAgent(
    "summary_agent",
    model_client=model_client,
    system_message="You are a customer support summarizer. Combine the responses from product_agent, shipping_agent, and returns_agent into a single, clear, friendly response for the customer. End your message with TERMINATE."
)

termination = TextMentionTermination("TERMINATE")
team = SelectorGroupChat(
    participants=[product_agent, shipping_agent, returns_agent, summary_agent],
    model_client=model_client,
    termination_condition=termination,
    allow_repeated_speaker=False,
    model_context=BufferedChatCompletionContext(buffer_size=5),
)

## Execution & Benchmarking

We define a `run_query` harness. This streams the group chat to the console and cleanly sums our performance metric so we can observe the total retrieval latency across all cascaded agent calls.

In [18]:
async def run_query(prompt: str):
    try:
        await Console(team.run_stream(task=prompt))
        
        # Aggregate Latencies
        valid_latencies = [l for l in latencies if l is not None]
        if valid_latencies:
            total_retrieval_ms = sum(valid_latencies)
            call_count = len(valid_latencies)
            total_display = "N/A" if total_retrieval_ms is None else ("< 1" if total_retrieval_ms == 0 else total_retrieval_ms)
            print(f"\n=> [Metrics] Made {call_count} Moss Retrievel queries. Total cumulative retrieval time: {total_display} ms")
            
    finally:
        # Ensuring the mutable list clears even if the stream raises an exception
        latencies.clear()
        # Reset the AutoGen conversation state for the next completely isolated query
        await team.reset()



In [19]:
# Query 1: Single-agent (product only)
await run_query("Is the Ergonomic Office Chair currently in stock and what are its specs?")

---------- TextMessage (user) ----------
Is the Ergonomic Office Chair currently in stock and what are its specs?
---------- ToolCallRequestEvent (product_agent) ----------

[Moss Retrieval Time] product_search context retrieved in 1 ms
[FunctionCall(id='call_QQYkPPPHLmTJyc0fpDjmcj7X', arguments='{"search_query":"Ergonomic Office Chair"}', name='product_search')]
---------- ToolCallExecutionEvent (product_agent) ----------
[FunctionExecutionResult(content='id: SKU-002, name: Ergonomic Office Chair, price: 349.99, stock: 0, specs: Lumbar support, adjustable armrests, mesh back, category: Furniture\n\nid: SKU-009, name: LED Desk Lamp, price: 34.99, stock: 150, specs: Adjustable color temperature, USB charging port, category: Furniture\n\nid: SKU-005, name: Yoga Mat, price: 49.99, stock: 60, specs: 6mm thick, non-slip, eco-friendly TPE, category: Fitness\n\nid: SKU-004, name: Mechanical Keyboard, price: 129.99, stock: 15, specs: TKL layout, Cherry MX Red switches, RGB backlight, category:

In [20]:
# Query 2: Single-agent (shipping only)
await run_query("What are my shipping options and how much does express delivery cost?")

---------- TextMessage (user) ----------
What are my shipping options and how much does express delivery cost?
---------- ToolCallRequestEvent (shipping_agent) ----------

[Moss Retrieval Time] shipping_search context retrieved  in < 1 ms
[FunctionCall(id='call_S0t2eQEGmU4dkay3lhTOmG5b', arguments='{"search_query":"shipping options and express delivery cost"}', name='shipping_search')]
---------- ToolCallExecutionEvent (shipping_agent) ----------
[FunctionExecutionResult(content='type: Express Shipping, cost: 14.99, eta_days: 1-2, eligible_for: In-stock items only\n\ntype: Heavy Freight Delivery, cost: 49.99, eta_days: 7-14, eligible_for: Furniture and heavy fitness equipment\n\ntype: Standard Shipping, cost: 4.99, eta_days: 5-7, eligible_for: All orders\n\ntype: Free Shipping, cost: 0, eta_days: 5-7, eligible_for: Orders over $75\n\ntype: Same-Day Delivery, cost: 29.99, eta_days: 0, eligible_for: Local zip codes, placed before 12 PM\n\ntype: Damaged Item Replacement Shipping, cost: 0,

In [21]:
# Query 3: Multi-agent Coordinated (returns -> shipping -> summary)
await run_query("My headphones arrived damaged. Can I get a replacement and how quickly can it be shipped?")

---------- TextMessage (user) ----------
My headphones arrived damaged. Can I get a replacement and how quickly can it be shipped?
---------- ToolCallRequestEvent (returns_agent) ----------

[Moss Retrieval Time] returns_search context retrieved in < 1 ms
[FunctionCall(id='call_yky9byxnDRdmFvV3C72SXFMU', arguments='{"search_query":"damaged headphones replacement policy"}', name='returns_search')]
---------- ToolCallExecutionEvent (returns_agent) ----------
[FunctionExecutionResult(content='policy: Damaged Item Return, window_days: 90, condition: Item damaged on arrival or within 90 days of purchase, refund_method: Full refund or free replacement, processing_days: 2\n\npolicy: Fitness Equipment Return, window_days: 30, condition: Unused and in original packaging, refund_method: Full refund, processing_days: 5\n\npolicy: Furniture Return, window_days: 14, condition: Unassembled and in original packaging only, refund_method: Store credit only, processing_days: 10\n\npolicy: Electronics Re

In [23]:
# Query 4: Complex Flow to force multiple cascading tool calls
prompt = """I am placing a massive order for our new office but need to verify a few critical things. I am buying an Ergonomic Office Chair, a Mechanical Keyboard, and a Yoga Mat. 

1. For the chair, does it have lumbar support? 
2. For the keyboard, what type of switches does it have? 
3. For the mat, is it eco-friendly? 
4. If the keyboard arrives completely defective, what is the exact return policy and timeline for getting a replacement? 
5. For the replacement, can it be shipped via Express Shipping? 
6. Finally, please break down the shipping costs for Express vs Standard, and tell me if my total combined order qualifies for Free Shipping based on the prices of those three items."""
await run_query(prompt)

---------- TextMessage (user) ----------
I am placing a massive order for our new office but need to verify a few critical things. I am buying an Ergonomic Office Chair, a Mechanical Keyboard, and a Yoga Mat. 

1. For the chair, does it have lumbar support? 
2. For the keyboard, what type of switches does it have? 
3. For the mat, is it eco-friendly? 
4. If the keyboard arrives completely defective, what is the exact return policy and timeline for getting a replacement? 
5. For the replacement, can it be shipped via Express Shipping? 
6. Finally, please break down the shipping costs for Express vs Standard, and tell me if my total combined order qualifies for Free Shipping based on the prices of those three items.
---------- ToolCallRequestEvent (product_agent) ----------

[Moss Retrieval Time] product_search context retrieved in < 1 ms

[Moss Retrieval Time] product_search context retrieved in < 1 ms

[Moss Retrieval Time] product_search context retrieved in < 1 ms
[FunctionCall(id='c